In [ ]:
#--- Database connection test ---
from sqlalchemy import create_engine, inspect
from dotenv import load_dotenv
import os

load_dotenv()

engine = create_engine(
    "postgresql+psycopg2://",
    connect_args={x
        "host":     os.getenv("DB_HOST"),
        "port":     os.getenv("DB_PORT"),
        "dbname":   os.getenv("DB_NAME"),
        "user":     os.getenv("DB_USER"),
        "password": os.getenv("DB_PASSWORD"),
    }
)

# Test the connection
with engine.connect() as conn:
    print("✅ Connected to PostgreSQL successfully!")

In pgadmin, created primary keys, foreign keys and fact tables are indexed.
Before creating foreign keys index the fact table.
#### Primary keys
1. dim_branch[branch_code]
2. dim_acc[acc_no]
#### bank_fact
1. index - idx_bank_fact_acc_branch
2. constraint - fk_bank_account

#### ledger_fact
1. index - idx_ledger_fact_acc_branch
2. constrain - fk_ledger_account

In [ ]:
#--- Create bank_fact table ---
from sqlalchemy import MetaData, Table, Column, String, Numeric, Date, text, UniqueConstraint

metadata = MetaData()

bank_fact = Table(
    "bank_fact",
    metadata,

    Column("txn_date", Date),                 
    Column("cheque_no", String(50)),
    Column("particulars", String(255)),
    Column("cust_name", String(150)),

    Column("credit_amount", Numeric(15, 2)),
    Column("debit_amount", Numeric(15, 2)),
    Column("balance_amount", Numeric(15, 2)),

    Column("value_date", Date),

    Column("branch_code", String(20)),
    Column("branch_name", String(100)),
    Column("acc_no", String(30)),

    Column("txn_type", String(20)),
    Column("label", String(20)),
)

metadata.create_all(engine)

In [ ]:
#--- Create bank_balance table ---
bank_balance = Table(
    "bank_balance",
    metadata,

   
    Column("branch_code", String(20)),
    Column("branch_name", String(100)),
    Column("acc_no", String(30), primary_key=True),
    Column("opening_balance", Numeric(15, 2)),
    Column("closing_balance", Numeric(15, 2)),
    Column("label", String(20)),

    UniqueConstraint("branch_code", "acc_no", name="uq_branch_acc")
)

# Create table
metadata.create_all(engine)

In [ ]:
#--- Create ledger_balance table ---
from sqlalchemy import MetaData, Table, Column, String, Numeric, Date, text, UniqueConstraint

metadata = MetaData()

ledger_balance = Table(
    "ledger_balance",
    metadata,

    # Column("branch_id", String(50), primary_key=True, server_default=text("gen_random_uuid()")),  # Unique ID for each balance record
    Column("branch_code", String(20)),
    Column("branch_name", String(100)),
    Column("acc_no", String(30), primary_key=True),
    Column("opening_balance", Numeric(15, 2)),
    Column("closing_balance", Numeric(15, 2)),
    Column("label", String(20)),

    UniqueConstraint("branch_code", "acc_no", name="uq_branch_acc")
)

# Create table
metadata.create_all(engine)